In [1]:
import warnings
warnings.filterwarnings('ignore')

In [1]:
import torch
import torch.nn as nn
import pandas as pd

In [26]:
df = pd.DataFrame({
    "input": [
        "I love machine",
        "I love deep",
        "I like machine",
        "I like deep",
        "machine learning is",
        "deep learning is",
        "Python is very",
        "PyTorch is very",
        "AI is crucial for our",
        "GRU is very"
    ],
    "target": [
        "learning",
        "learning",
        "learning",
        "learning",
        "powerful",
        "powerful",
        "useful",
        "useful",
        "task",
        "powerful"
    ]
})
df.shape

(10, 2)

In [ ]:
def tokenize(text):
    return text.split()

In [18]:
vocab = {
    "<unk>": 0
}

def build_vocab(df):
    input = tokenize(df['input'])
    target = tokenize(df['target'])

    sentences = input + target

    for word in sentences:
        if word not in vocab:
            vocab[word] = len(vocab)

df.apply(build_vocab, axis=1)

0    None
1    None
2    None
3    None
4    None
5    None
6    None
7    None
8    None
9    None
dtype: object

In [19]:
len(vocab)

19

In [24]:
def textToNumeric(text, vocab):
    return torch.tensor(
        [vocab.get(word, vocab["<unk>"]) for word in tokenize(text)],
        dtype=torch.long
    )

In [25]:
print(textToNumeric("machine learning is powerful", vocab))

tensor([3, 4, 7, 8])


In [31]:
from torch.utils.data import Dataset, DataLoader

In [32]:
class NextToken(Dataset):
    def __init__(self, df, vocab):
        self.data = df
        self.vocab = vocab

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, index):
        input = textToNumeric(self.data.iloc[index]['input'], self.vocab)
        target = textToNumeric(self.data.iloc[index]['target'], self.vocab)

        return input, target

In [35]:
dataset = NextToken(df, vocab)

In [36]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [37]:
for question, answer in dataloader:
    print(question, answer)

tensor([[18,  7, 10]]) tensor([[8]])
tensor([[1, 2, 3]]) tensor([[4]])
tensor([[13,  7, 14, 15, 16]]) tensor([[17]])
tensor([[1, 6, 5]]) tensor([[4]])
tensor([[5, 4, 7]]) tensor([[8]])
tensor([[3, 4, 7]]) tensor([[8]])
tensor([[ 9,  7, 10]]) tensor([[11]])
tensor([[12,  7, 10]]) tensor([[11]])
tensor([[1, 2, 5]]) tensor([[4]])
tensor([[1, 6, 3]]) tensor([[4]])


In [45]:
dataset[0][0]
x = nn.Embedding(len(vocab), 64)
x(dataset[0][0]).shape

torch.Size([3, 64])

In [62]:
class SimpleRNN(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim
        )

        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, vocab_size)


    def forward(self, text):
        emdedding = self.embedding(text)
        _, final = self.rnn(emdedding)
        output = self.fc(final.squeeze(0))
        return output

In [63]:
learning_rate = 0.01
epochs = 5

In [64]:
model = SimpleRNN(len(vocab), 32, 64)

In [65]:
loss_fn = nn.CrossEntropyLoss()
optim = torch.optim.RMSprop(model.parameters(), lr=learning_rate)

In [66]:
for epoch in range(epochs):
    for question, target in dataloader:
        optim.zero_grad()
        predict = model(question)
        loss = loss_fn(predict, target[0])
        loss.backward()
        optim.step()

    print(f"Epochs: {epoch + 1}, Loss: {loss.item()}")

Epochs: 1, Loss: 2.224259376525879
Epochs: 2, Loss: 0.003062085248529911
Epochs: 3, Loss: 2.9083352088928223
Epochs: 4, Loss: 0.0038517348002642393
Epochs: 5, Loss: 0.07957617938518524


In [81]:
def predict(model, question):
    contect_vec = textToNumeric(question, vocab)
    vec = contect_vec.unsqueeze(0)
    output = model(vec)
    probabilities = torch.nn.functional.softmax(output, dim=1)
    _, index = torch.max(probabilities, dim=1)
    return list(vocab)[index]

In [85]:
predict(model, "GUR is")

'AI'